# Lab6.2 — Validarea overlay-ului FPGA Resize → Grayscale → Sobel

**Placă:** PYNQ-Z2 · **PYNQ:** 2.7.0 · **Vivado/Vitis HLS:** 2022.2

Scop: verificarea pe hardware real a pipeline-ului **DDR → AXI DMA → Resize (PL) → Grayscale + Sobel (PL) → AXI DMA → DDR**. Cele două acceleratoare comunică direct prin AXI4-Stream.

**Înainte de execuție:** copiați **ambele** fișiere `resize_sobel.bit` și `resize_sobel.hwh`, generate din același build Vivado, în directorul `Lab6/` de pe PYNQ-Z2, chiar lângă notebook (sau în `Lab6/overlay/`). Nu redenumiți numai unul dintre ele.

Executați pe rând primele **două celule de cod**, verificați rezultatul testului sintetic și abia apoi continuați la benchmark. Testele se bazează pe registrele confirmate din `resize_sobel.hwh`, furnizat după implementarea Vivado 2022.2.

**Atenție:** după un timeout DMA, reporniți kernelul Jupyter înainte de o nouă încercare. Afișarea unei imagini nu înseamnă că rezultatul este și numeric corect; comparați întotdeauna cu referința software.

## 1. Încărcarea overlay-ului și configurarea transferului DMA

Din `overlay.ip_dict` și `register_map` identificăm automat acceleratoarele și registrele acestora. Nu folosim adrese fizice hard-codate.

Traseul AXI4-Stream este **DMA 32 biți → convertor 24 biți → Resize 24 biți → Sobel 24→8 biți → convertor 32 biți → DMA**.

Funcția de test utilizează `ap_start = 0x01`, **fără auto-restart**, și verifică terminarea ambelor canale DMA cu timeout.

In [ ]:
from pathlib import Path
from time import perf_counter, sleep
import numpy as np
import cv2
from pynq import Overlay, allocate

def locate_fused_bit():
    here = Path.cwd()
    possibilities = [
        here / "overlay" / "resize_sobel.bit",
        here / "resize_sobel.bit",
        here.parent / "Lab6" / "overlay" / "resize_sobel.bit",
        Path("/home/xilinx/jupyter_notebooks/Edge-Computing/Lab6/overlay/resize_sobel.bit"),
    ]
    for bitfile in possibilities:
        if bitfile.is_file() and bitfile.with_suffix(".hwh").is_file():
            return bitfile
    raise FileNotFoundError(
        "Copiați resize_sobel.bit și resize_sobel.hwh (aceeași versiune) "
        "în Edge-Computing/Lab6/ (lângă notebook) sau Lab6/overlay/ de pe PYNQ-Z2."
    )

BITFILE = locate_fused_bit()
overlay = Overlay(str(BITFILE))
required = ("axi_dma_0", "resize_accel_0", "sobel_accel_0")
missing = [name for name in required if name not in overlay.ip_dict]
if missing:
    raise RuntimeError("Lipsesc componentele din HWH: " + ", ".join(missing))

dma = overlay.axi_dma_0
resize_ip = overlay.resize_accel_0
sobel_ip = overlay.sobel_accel_0

print("Overlay încărcat:", BITFILE)
for name in required:
    print(name, overlay.ip_dict[name]["type"])
print("Registre Resize:", resize_ip.register_map)
print("Registre Sobel:", sobel_ip.register_map)

def wait_dma(channel, label, timeout_s=5.0):
    t0 = perf_counter()
    while not channel.idle:
        if channel.error:
            raise RuntimeError("DMA {}: eroare; reporniți kernelul Jupyter.".format(label))
        if perf_counter() - t0 >= timeout_s:
            raise TimeoutError(
                "DMA {}: timeout {:.1f}s; reporniți kernelul Jupyter înainte "
                "de o nouă încercare.".format(label, timeout_s))
        sleep(0.001)
    channel.wait()

class FusedResizeSobel:
    def __init__(self, dma, resize_ip, sobel_ip):
        self.dma, self.resize_ip, self.sobel_ip = dma, resize_ip, sobel_ip
        self.geometry = None
        self.buf_in, self.buf_out = None, None

    def close(self):
        for name in ("buf_in", "buf_out"):
            buf = getattr(self, name)
            if buf is not None:
                buf.freebuffer()
                setattr(self, name, None)
        self.geometry = None

    def run(self, image_bgr, size):
        if image_bgr.dtype != np.uint8 or image_bgr.ndim != 3 or image_bgr.shape[2] != 3:
            raise ValueError("Intrarea trebuie să fie imagine BGR uint8 cu 3 canale.")
        h, w = image_bgr.shape[:2]
        out_w, out_h = map(int, size)
        if not (1 <= out_w <= 1920 and 1 <= out_h <= 1080 and
                1 <= w <= 3840 and 1 <= h <= 2160):
            raise ValueError("Dimensiuni în afara limitelor IP-urilor HLS.")
        if (h * w * 3) % 4 or (out_h * out_w) % 4:
            raise ValueError("Pentru primul test DMA, lungimile în octeți "
                             "trebuie să fie multipli de 4 (DRE=0).")
        geom = (h, w, out_h, out_w)
        if geom != self.geometry:
            self.close()
            self.buf_in = allocate(shape=(h, w, 3), dtype=np.uint8)
            self.buf_out = allocate(shape=(out_h, out_w), dtype=np.uint8)
            self.geometry = geom

        np.copyto(self.buf_in, np.ascontiguousarray(image_bgr))
        self.buf_in.flush()
        self.buf_out.fill(0)
        self.buf_out.flush()

        rr = self.resize_ip.register_map
        rr.src_rows, rr.src_cols = h, w
        rr.dst_rows, rr.dst_cols = out_h, out_w
        sr = self.sobel_ip.register_map
        sr.rows, sr.cols = out_h, out_w

        # Mai întâi pregătim receptorul, apoi nucleul din aval, apoi producătorul.
        self.dma.recvchannel.transfer(self.buf_out)
        self.sobel_ip.write(0x00, 0x01)   # ap_start, one-shot
        self.resize_ip.write(0x00, 0x01)  # ap_start, one-shot
        self.dma.sendchannel.transfer(self.buf_in)

        wait_dma(self.dma.sendchannel, "MM2S")
        wait_dma(self.dma.recvchannel, "S2MM")
        self.buf_out.invalidate()
        return np.array(self.buf_out, copy=True)

hw = FusedResizeSobel(dma, resize_ip, sobel_ip)
print("Driver pregătit. Urmează testul sintetic 64×48 → 32×24.")

## 2. Testul sintetic pe FPGA

Imaginea conține gradienți pe cele trei canale BGR, astfel încât **ieșirea Sobel nu ar trebui să fie uniform zero**.

Rezultatul FPGA este comparat cu secvența OpenCV: Resize biliniar → Grayscale → Sobel L2, margini `BORDER_REFLECT_101`. Grayscale-ul HLS utilizează coeficienți întregi, iar Resize HLS folosește rotunjiri proprii; prin urmare nu impunem identitate exactă pixel cu pixel.

**După această celulă opriți-vă și transmiteți rezultatul (MAE, diferența maximă și pixelii diferiți).**

In [ ]:
from matplotlib import pyplot as plt

src_h, src_w = 48, 64
ys, xs = np.indices((src_h, src_w))
test_bgr = np.empty((src_h, src_w, 3), dtype=np.uint8)
test_bgr[:, :, 0] = (4 * xs + 2 * ys) % 256
test_bgr[:, :, 1] = (7 * xs + 3 * ys) % 256
test_bgr[:, :, 2] = ((xs // 8) ^ (ys // 6)) * 40
target = (32, 24)

output_fpga = hw.run(test_bgr, target)
small_cpu = cv2.resize(test_bgr, target, interpolation=cv2.INTER_LINEAR)
gray_cpu = cv2.cvtColor(small_cpu, cv2.COLOR_BGR2GRAY)
gx = cv2.Sobel(gray_cpu, cv2.CV_32F, 1, 0, ksize=3,
               borderType=cv2.BORDER_REFLECT_101)
gy = cv2.Sobel(gray_cpu, cv2.CV_32F, 0, 1, ksize=3,
               borderType=cv2.BORDER_REFLECT_101)
reference = cv2.convertScaleAbs(cv2.magnitude(gx, gy))
diff = np.abs(output_fpga.astype(np.int16) - reference.astype(np.int16))

print("Dimensiunea FPGA:", output_fpga.shape, "| dtype:", output_fpga.dtype)
print("MAE:", float(diff.mean()))
print("Diferența maximă:", int(diff.max()))
print("Fracțiune pixeli diferiți:", float(np.mean(diff != 0)))
print("Min/max FPGA:", int(output_fpga.min()), int(output_fpga.max()))
print("Pixeli FPGA nenuli:", int(np.count_nonzero(output_fpga)))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].imshow(reference, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("OpenCV: referință")
axes[1].imshow(output_fpga, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("FPGA: Resize → Sobel")
axes[2].imshow(diff, cmap="inferno")
axes[2].set_title("Diferență absolută")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Benchmark comparativ — numai după validarea testului sintetic

Măsurăm timpul complet **al apelului de procesare**, inclusiv copierea datelor, lansarea acceleratoarelor și DMA; **nu** include captură video. Comparația este cu traseul ARM Resize → Grayscale → Sobel pe aceeași imagine. Nu presupuneți că un speed-up este garantat.

**Nu executați această celulă până când testul sintetic nu s-a încheiat corect.**

In [ ]:
from lab6_common import load_test_image, process_cpu, measure_callable, compare_images

image = load_test_image()
sizes = [(160, 120), (256, 192), (320, 240), (480, 320)]
rows = []
for size in sizes:
    # Testăm separat validitatea transferului înainte de cronometrare.
    try:
        ref = process_cpu(image, size)
        got = hw.run(image, size)
        quality = compare_images(ref, got)
        arm = measure_callable(lambda: process_cpu(image, size), repeats=15)
        fused = measure_callable(lambda: hw.run(image, size), repeats=15)
        rows.append((size, arm["mean_ms"], fused["mean_ms"], quality))
    except Exception as exc:
        print("EROARE la {}: {}".format(size, exc))
        print("Dacă este timeout DMA, reporniți kernelul Jupyter.")
        raise

print("{:<12s} {:>12s} {:>12s} {:>10s} {:>10s} {:>8s}".format(
    "Rezoluție", "ARM (ms)", "FPGA (ms)", "Speed-up", "MAE", "MaxDiff"))
for size, t_arm, t_fpga, q in rows:
    print("{:<12s} {:>12.3f} {:>12.3f} {:>10.3f} {:>10.3f} {:>8d}".format(
        "{}x{}".format(*size), t_arm, t_fpga, t_arm / t_fpga,
        q["mae"], q["max_abs"]))

fig, ax = plt.subplots(figsize=(8, 4))
labels = ["{}x{}".format(*row[0]) for row in rows]
ax.plot(labels, [row[1] for row in rows], "o-", label="ARM")
ax.plot(labels, [row[2] for row in rows], "s-", label="FPGA Resize+Sobel")
ax.set_ylabel("Timp procesare (ms)")
ax.set_title("Lab6.2 — procesare ARM vs FPGA (fără captură video)")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

## Interpretare și precauții

1. `MAE`, `max_abs` și diferența vizuală caracterizează corectitudinea numerică față de OpenCV, nu performanța.
2. Benchmark-ul include lansarea nucleelor și DMA, dar nu capturarea unui cadru de la cameră.
3. Speed-up sub 1 înseamnă că FPGA a fost mai lent în configurația respectivă.
4. Dacă DMA se blochează, nu continuați benchmark-ul în același kernel Jupyter. Examinați jurnalul, geometria, `TLAST` și dimensiunile transferurilor.
5. Păstrați obiectul hw pentru variantele B și C; ultima celulă a variantei C eliberează bufferele.

## Varianta B — imagini din fișiere (JPG, PNG)

Folosește overlay-ul încărcat la începutul notebook-ului (varianta A). Pentru fiecare imagine: citire de pe disc, normalizare BGR 640×480, comparație OpenCV ARM / FPGA, verificare MAE și benchmark.

Imaginile de intrare pot fi adăugate în Lab6/images. Citirea și normalizarea sunt excluse din benchmark, iar CPU și FPGA primesc aceiași pixeli. Executați această secțiune numai după validarea testului A.

In [ ]:
from lab6_common import process_cpu, compare_images, measure_callable

IMAGE_FILES = [Path("../Lab4/images/fruits.jpg")]
# Adăugați imagini: Path("images/exemplu.jpg"), Path("images/exemplu.png")
INPUT_SIZE = (640, 480)
OUTPUT_SIZE = (320, 240)
REPEATS = 15

def normalize_bgr(image, size=INPUT_SIZE):
    if image is None or image.ndim != 3 or image.shape[2] != 3 or image.dtype != np.uint8:
        raise ValueError("Este necesară o imagine BGR uint8.")
    if (image.shape[1], image.shape[0]) != size:
        return cv2.resize(image, size, interpolation=cv2.INTER_LINEAR)
    return np.ascontiguousarray(image)

results_b = []
for file in IMAGE_FILES:
    file = Path(file)
    if not file.is_file():
        raise FileNotFoundError("Lipsește imaginea: {}. Încărcați-o în Lab6/images/.".format(file))
    frame = cv2.imread(str(file), cv2.IMREAD_COLOR)
    if frame is None:
        raise RuntimeError("OpenCV nu poate citi imaginea: " + str(file))
    frame = normalize_bgr(frame)
    arm = process_cpu(frame, OUTPUT_SIZE)
    pl = hw.run(frame, OUTPUT_SIZE)
    quality = compare_images(arm, pl)
    diff = np.abs(arm.astype(np.int16) - pl.astype(np.int16)).astype(np.uint8)
    if pl.dtype != np.uint8 or pl.shape != (OUTPUT_SIZE[1], OUTPUT_SIZE[0]):
        raise RuntimeError("Dimensiune sau format greșit la ieșirea FPGA.")
    print("\nImagine:", file, "| DMA:", frame.shape, "| ieșire:", pl.shape)
    print("MAE: {:.3f} | max: {} | fracție diferită: {:.2%} | nenuli FPGA: {}".format(
        quality["mae"], quality["max_abs"], quality["fraction_nonzero"], np.count_nonzero(pl)))
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
    images = [cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), arm, pl, diff]
    titles = ["Imagine intrare", "ARM OpenCV", "FPGA Resize-Sobel", "Diferența absolută"]
    for i, ax in enumerate(axes):
        ax.imshow(images[i], **({} if i == 0 else {"cmap": "gray", "vmin": 0, "vmax": 255}))
        ax.set_title(titles[i])
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    arm_t = measure_callable(lambda: process_cpu(frame, OUTPUT_SIZE), repeats=REPEATS)
    pl_t = measure_callable(lambda: hw.run(frame, OUTPUT_SIZE), repeats=REPEATS)
    print("ARM: {:.2f} ms | FPGA: {:.2f} ms | raport ARM/FPGA: {:.2f}x".format(
        arm_t["mean_ms"], pl_t["mean_ms"], arm_t["mean_ms"] / pl_t["mean_ms"]))
    results_b.append((str(file), quality, arm_t, pl_t))
print("Varianta B: {} imagini procesate.".format(len(results_b)))

## Varianta C — video din fișier sau cameră USB

Această secțiune folosește același overlay FPGA. Selectați MODE="file" pentru un fișier video din Lab6 (de exemplu video.mp4) sau MODE="camera" pentru o cameră USB, cu CAMERA_INDEX=0.

Etape: validăm primul cadru față de OpenCV, apoi măsurăm separat citire/decode, normalizare 640×480, procesare și decizie (densitatea muchiilor). FPS-ul observat este pentru bucla completă, fără randare sau salvare video. La final, opțional salvăm un fișier AVI/MJPEG cu rezultatele FPGA; exportul nu intră în benchmark.

Cele două teste ARM/FPGA redeschid sursa. Pentru fișier sunt aceleași cadre; pentru cameră sunt cadre preluate în momente diferite. După orice timeout DMA, reporniți kernelul.

In [ ]:
MODE = "file"          # "file" sau "camera"
VIDEO_FILE = Path("video.mp4")  # copiați fișierul în Lab6/
CAMERA_INDEX = 0
N_FRAMES = 50
INPUT_VIDEO = (640, 480)
OUTPUT_VIDEO = (320, 240)
EDGE_THRESHOLD = 80
SAVE_VIDEO = True

def open_source():
    if MODE == "file":
        if not VIDEO_FILE.is_file():
            raise FileNotFoundError("Lipsește {}. Încărcați video.mp4 în Lab6 sau selectați MODE='camera'.".format(VIDEO_FILE))
        cap = cv2.VideoCapture(str(VIDEO_FILE))
    elif MODE == "camera":
        cap = cv2.VideoCapture(CAMERA_INDEX)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, INPUT_VIDEO[0])
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, INPUT_VIDEO[1])
    else:
        raise ValueError("MODE trebuie să fie file sau camera")
    if not cap.isOpened():
        cap.release()
        raise RuntimeError("Nu se poate deschide sursa video")
    return cap

cap = open_source()
try:
    ok, original_c = cap.read()
finally:
    cap.release()
if not ok:
    raise RuntimeError("Sursa video nu furnizează cadre.")
frame_c = normalize_bgr(original_c, INPUT_VIDEO)
arm_c = process_cpu(frame_c, OUTPUT_VIDEO)
fpga_c = hw.run(frame_c, OUTPUT_VIDEO)
quality_c = compare_images(arm_c, fpga_c)
print("Validare video: MAE={:.3f} | MaxDiff={} | FPGA nenuli={}".format(
    quality_c["mae"], quality_c["max_abs"], np.count_nonzero(fpga_c)))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, im, title in zip(axes,
    [cv2.cvtColor(frame_c, cv2.COLOR_BGR2RGB), arm_c, fpga_c],
    ["Cadru video", "ARM OpenCV", "FPGA Resize-Sobel"]):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def measure_video(label, process):
    cap = open_source()
    times = []
    start = perf_counter()
    try:
        for i in range(N_FRAMES):
            t0 = perf_counter()
            ok, frame = cap.read()
            t1 = perf_counter()
            if not ok:
                break
            frame = normalize_bgr(frame, INPUT_VIDEO)
            t2 = perf_counter()
            edge = process(frame)
            t3 = perf_counter()
            fraction = float(np.mean(edge >= EDGE_THRESHOLD))
            t4 = perf_counter()
            times.append([1000*(t1-t0), 1000*(t2-t1), 1000*(t3-t2),
                          1000*(t4-t3), 1000*(t4-t0), fraction])
    finally:
        cap.release()
    elapsed = perf_counter() - start
    if not times:
        raise RuntimeError("Nu am putut procesa niciun cadru pentru " + label)
    a = np.asarray(times)
    return dict(label=label, n=len(times), capture=float(a[:,0].mean()),
                prepare=float(a[:,1].mean()), processing=float(a[:,2].mean()),
                decision=float(a[:,3].mean()), e2e=float(a[:,4].mean()),
                p95=float(np.percentile(a[:,4],95)), fps=len(times)/elapsed)

arm_report = measure_video("ARM", lambda image: process_cpu(image, OUTPUT_VIDEO))
pl_report = measure_video("FPGA", lambda image: hw.run(image, OUTPUT_VIDEO))
print("{:<6} {:>6} {:>10} {:>10} {:>10} {:>10} {:>10} {:>8}".format(
      "Cale", "Cadre", "Citire ms", "Prep ms", "Proces ms", "Decizie ms", "E2E ms", "FPS"))
for r in [arm_report, pl_report]:
    print("{:<6} {:>6} {:>10.2f} {:>10.2f} {:>10.2f} {:>10.2f} {:>10.2f} {:>8.2f}".format(
        r["label"],r["n"],r["capture"],r["prepare"],r["processing"],r["decision"],r["e2e"],r["fps"]))
    print("  p95 end-to-end = {:.2f} ms".format(r["p95"]))
print("Raport timp de procesare ARM/FPGA: {:.2f}x".format(
    arm_report["processing"]/pl_report["processing"]))
print("Raport FPS FPGA/ARM: {:.2f}x".format(pl_report["fps"]/arm_report["fps"]))

In [ ]:
OUTPUT_FILE = Path("Lab6.2C_FPGA_output.avi")
if SAVE_VIDEO:
    cap = open_source()
    writer = None
    count = 0
    try:
        source_fps = cap.get(cv2.CAP_PROP_FPS)
        fps = float(source_fps) if np.isfinite(source_fps) and 1 <= source_fps <= 60 else 15.0
        writer = cv2.VideoWriter(str(OUTPUT_FILE),
                                 cv2.VideoWriter_fourcc(*"MJPG"),
                                 fps, OUTPUT_VIDEO, True)
        if not writer.isOpened():
            print("Encoderul MJPEG nu este disponibil. Benchmark-ul rămâne valabil.")
        else:
            for _ in range(N_FRAMES):
                ok, frame = cap.read()
                if not ok:
                    break
                edges = hw.run(normalize_bgr(frame, INPUT_VIDEO), OUTPUT_VIDEO)
                writer.write(cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR))
                count += 1
            print("Video procesat:", OUTPUT_FILE.resolve(), "| cadre:", count)
    finally:
        cap.release()
        if writer is not None:
            writer.release()
# Eliberați bufferele după toate experimentele A, B și C.
hw.close()
print("Lab6.2: experimente finalizate.")